<a href="https://colab.research.google.com/github/codey-m/prob_stats/blob/main/module3_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 Lab: Measure & Relate
## 6.3710x — Probability and Statistical Data Analysis

### Where We Left Off

In Module 2, you discretized the FL/CL ratio into bins, built empirical PMFs, and discovered that male and female crabs might have different ratio distributions. But you were limited: only 100 crabs, only 2 measurements, and everything was discrete.

### What's Changed

1. **More crabs.** You now have 200 — the full sample.
2. **More measurements.** Three new body dimensions: rear width (RW), carapace width (CW), and body depth (BD).
3. **No more binning.** You'll work with continuous distributions directly.

And there's still something about these crabs we haven't told you.

### The Plan

We'll meet all five measurements, then **focus on three — FL, CL, and RW** — because three dimensions is the boundary of what we can visualize. You'll see how covariance, eigenstructure, and conditioning work in 2D (where geometry is clearest) and then extend each idea to 3D (where the scaling becomes visible). Everything you learn generalizes to higher dimensions. Module 4 will take you there.

### What's New Conceptually

- **Joint distributions** across multiple continuous variables
- **Covariance and correlation** to quantify relationships
- **The covariance matrix $\boldsymbol{\Sigma}$** and its eigenstructure — in 2D and 3D
- **Conditioning** — slicing the ellipse, then slicing the ellipsoid. Same block formula, bigger matrices.
- **Bayesian estimation** of the mean ratio — and the connection between Bayes' rule and the block conditioning formula


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from mpl_toolkits.mplot3d import Axes3D

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11

def load_data(filename):
    """Load CSV from local directory or GitHub."""
    if os.path.exists(filename):
        return pd.read_csv(filename)
    base_url = ("https://raw.githubusercontent.com/"
                "codey-m/prob-stats-labs/main/")
    url = base_url + filename
    print(f"Loading from GitHub: {filename}")
    return pd.read_csv(url)

## Part 1: The Expanded Dataset

Let's load the Module 3 data and see what's new.


In [ ]:
crabs = load_data("crabs_module3.csv")

print("Shape:", crabs.shape)
print("Columns:", list(crabs.columns))
crabs.head()

In [ ]:
crabs.describe().round(2)

### TODO 1

How many crabs are in this dataset? What new columns appear that weren't in Module 2? Compute a new column called `ratio` equal to FL / CL.


In [ ]:
# TODO 1: Your code here
n_crabs = ...
print(f"Number of crabs: {n_crabs}")

# Compute ratio
crabs["ratio"] = ...

print(f"\nRatio mean:  {crabs['ratio'].mean():.4f}")
print(f"Ratio std:   {crabs['ratio'].std():.4f}")

## Part 2: The Full Picture

You now have five body measurements per crab: FL, RW, CL, CW, BD. Let's see how they all relate before we focus in.

A **scatterplot matrix** shows every pairwise combination. Each cell is a scatter plot of two variables. The diagonal shows the distribution of each variable individually.


In [ ]:
measurements = ["FL", "RW", "CL", "CW", "BD"]

sns.pairplot(crabs[measurements], corner=True,
             plot_kws={"alpha": 0.4, "s": 15,
                       "color": "steelblue"},
             diag_kws={"color": "steelblue"})
plt.suptitle("Pairwise Scatterplots of Body Measurements",
             y=1.02, fontsize=14, fontweight="bold")
plt.show()

Let's reflect on our scatterplot matrix:

- Which pair of variables appears MOST tightly clustered (strongest relationship)?
- Which pair appears to have the MOST scatter (weakest relationship)?
- Do you see any hints of subgroups — clusters or bands — in any of the plots?


### TODO 2a

Compute the **correlation matrix** of the five measurements and visualize it as a heatmap.

Recall: correlation quantifies the strength of a linear relationship:

$$\rho_{X,Y} = \frac{\text{Cov}(X,Y)}{\sigma_X \, \sigma_Y}$$

It ranges from $-1$ (perfect negative) to $+1$ (perfect positive).


In [ ]:
# TODO 2a: Compute and print the correlation matrix
# Hint: Consider the pandas `corr` method

corr_matrix = ...

print("Correlation Matrix:")
print(corr_matrix.round(4))

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(corr_matrix, annot=True, fmt=".3f",
            cmap="RdYlBu_r", vmin=0.85, vmax=1.0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix of Body Measurements",
             fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

### TODO 2b

Now identify the **strongest** and **weakest** correlated pairs
of measurements.

In [ ]:
# TODO 2b: Find the strongest and weakest pairs of measurements
# Hint: iterate over all pairs using itertools.combinations

pairs = ...

pair_corrs = [(a, b, corr_matrix.loc[a, b]) for a, b in list(pairs)]
pair_corrs_sorted = sorted(pair_corrs, key=lambda x: x[2])

strongest = pair_corrs_sorted[-1]
weakest = pair_corrs_sorted[0]

print(f"Strongest correlation: {strongest[0]}-{strongest[1]}"
      f" = {strongest[2]:.4f}")
print(f"Weakest correlation:   {weakest[0]}-{weakest[1]}"
      f" = {weakest[2]:.4f}")

### TODO 2c

All correlations are very high — these measurements all scale with body size. But some variables carry more **unique information** than others.

Compute the **mean correlation** of each variable with the other four. Which variable is most redundant (highest mean correlation)? Which carries the most independent information (lowest mean correlation)?


In [ ]:
# TODO 2c: Your code here
# For each variable, compute its mean correlation with the
# other four.

print("Mean correlation of each variable with all others:")

mean_corrs = {}
for var in measurements:
    others = [v for v in measurements if v != var]
    mean_r = ...
    mean_corrs[var] = mean_r
    print(f"  {var:>2s}: {mean_r:.4f}")

most_redundant = max(mean_corrs, key=mean_corrs.get)
most_independent = min(mean_corrs, key=mean_corrs.get)

print(f"\nMost redundant:    {most_redundant}")
print(f"Most independent:  {most_independent}")

### Choosing Our Focus

All five variables are highly correlated because they all scale with body size. But **RW (rear width)** stands out — it has the lowest average correlation with the others, meaning it carries the most information not already captured by the length measurements.

For the rest of this module, we'll work with **FL, CL, and RW**: two length measurements and one width measurement. Three dimensions is the limit of what we can visualize — and it's enough to see how covariance, eigenstructure, and conditioning scale beyond 2D. Module 4 will handle all five with PCA.


## Part 3: The Covariance Matrix — Structure in 2D and 3D

In lecture 3.4, you learned that the covariance matrix $\boldsymbol{\Sigma}$ encodes all pairwise relationships, its **eigenvectors** define the natural axes of the distribution, and its **eigenvalues** tell you how much variance lives along each axis.

Let's build this up: first in 2D where the geometry is clearest, then in 3D where we can see the ideas scale.


### TODO 3a

Compute the 2×2 covariance matrix for FL and CL — the pair you know from Module 2.


In [ ]:
# TODO 3a: Covariance matrix for FL, CL
pair_data = crabs[["FL", "CL"]]
cov_2d = ...

print("Covariance matrix (FL, CL):")
print(cov_2d.round(4))

print(f"\nVar(FL)        = {cov_2d.loc['FL','FL']:.4f}")
print(f"Var(CL)        = {cov_2d.loc['CL','CL']:.4f}")
print(f"Cov(FL, CL)    = {cov_2d.loc['FL','CL']:.4f}")

### TODO 3b

Compute the eigenvalues and eigenvectors of this covariance matrix. What proportion of the total variance is explained by the first (largest) eigenvector?

*Hint:* `np.linalg.eigh` returns eigenvalues in ascending order. You'll want to reverse them.


In [ ]:
# TODO 3b: Eigendecomposition in 2D
Sigma_2d = cov_2d.values
eigenvalues_2d, eigenvectors_2d = np.linalg.eigh(Sigma_2d)

# Reverse to descending order
eigenvalues_2d = eigenvalues_2d[::-1]
eigenvectors_2d = eigenvectors_2d[:, ::-1]

# proportion of variance explained by the first eigenvalue
prop_first_2d = ...

print("2D Eigendecomposition (FL, CL):")
print(f"  λ₁ = {eigenvalues_2d[0]:.4f}  "
      f"({prop_first_2d:.1%} of variance)")
print(f"  λ₂ = {eigenvalues_2d[1]:.4f}  "
      f"({eigenvalues_2d[1]/eigenvalues_2d.sum():.1%} "
      f"of variance)")

print(f"\nEigenvectors:")
print(f"  v₁ = [{eigenvectors_2d[0,0]:.4f}, "
      f"{eigenvectors_2d[1,0]:.4f}]")
print(f"  v₂ = [{eigenvectors_2d[0,1]:.4f}, "
      f"{eigenvectors_2d[1,1]:.4f}]")

### Drawing the MVN Ellipse from Eigendecomposition

In lecture, you learned that the eigenvectors of $\boldsymbol{\Sigma}$ define the axes of the MVN ellipse, and the eigenvalues control how wide it is along each axis.

The function below turns those quantities into an ellipse we can draw. You don't need to modify it — just notice what it takes as input: the same eigenvalues and eigenvectors you just computed.

In [ ]:
from matplotlib.patches import Ellipse

def eigen_ellipse(mu, eigenvalues, eigenvectors,
                  n_std=2, **kwargs):
    """Create an MVN ellipse from eigendecomposition.

    Parameters
    ----------
    mu           : center (2,)
    eigenvalues  : [λ₁, λ₂] in descending order
    eigenvectors : columns are v₁, v₂
    n_std        : number of standard deviations (1, 2, or 3)
    **kwargs     : passed to matplotlib Ellipse (color, alpha, etc.)
    """
    angle = np.degrees(np.arctan2(eigenvectors[1, 0],
                                   eigenvectors[0, 0]))
    width = 2 * n_std * np.sqrt(eigenvalues[0])
    height = 2 * n_std * np.sqrt(eigenvalues[1])
    return Ellipse(xy=mu, width=width, height=height,
                   angle=angle, **kwargs)

### Visualize: Data + MVN Contour + Eigenvectors in 2D

Let's put it all together — the data points, the MVN contour ellipse, and the eigenvector arrows. This is the picture you can only draw cleanly in 2D.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Data points
ax.scatter(crabs["CL"], crabs["FL"], color="steelblue",
           alpha=0.4, s=30, edgecolor="white", linewidth=0.3)

# Ellipse from eigendecomposition
mu_2d = pair_data.mean().values
mu_plot = [mu_2d[1], mu_2d[0]]  # CL, FL

# Flip eigenvector components to (CL, FL) order for plotting
eigenvalues_plot = eigenvalues_2d
eigenvectors_plot = np.array([[eigenvectors_2d[1, 0], eigenvectors_2d[1, 1]],
                               [eigenvectors_2d[0, 0], eigenvectors_2d[0, 1]]])

for n_std in [1, 2, 3]:
    ellipse = eigen_ellipse(mu_plot, eigenvalues_plot,
                            eigenvectors_plot, n_std=n_std,
                            fill=False, color="steelblue",
                            linewidth=1.5, alpha=0.6)
    ax.add_patch(ellipse)

# Eigenvectors — scale by sqrt(eigenvalue) for visibility
colors_ev = ["#F44336", "#2196F3"]
labels_ev = [
    f"v₁: most variance (λ₁={eigenvalues_2d[0]:.2f})",
    f"v₂: least variance (λ₂={eigenvalues_2d[1]:.2f})"
]

for i in range(2):
    # Eigenvector in (FL, CL); flip to (CL, FL) for plotting
    vec = np.array([eigenvectors_2d[1, i],
                    eigenvectors_2d[0, i]])
    vec_scaled = vec * np.sqrt(eigenvalues_2d[i]) * 3

    ax.annotate("", xy=(mu_plot[0] + vec_scaled[0],
                        mu_plot[1] + vec_scaled[1]),
                xytext=(mu_plot[0], mu_plot[1]),
                arrowprops=dict(arrowstyle="->",
                                color=colors_ev[i], lw=3))
    ax.annotate("", xy=(mu_plot[0] - vec_scaled[0],
                        mu_plot[1] - vec_scaled[1]),
                xytext=(mu_plot[0], mu_plot[1]),
                arrowprops=dict(arrowstyle="->",
                                color=colors_ev[i], lw=3))

for i in range(2):
    ax.plot([], [], color=colors_ev[i], linewidth=3,
            label=labels_ev[i])

ax.set_xlabel("Carapace Length (mm)", fontsize=12)
ax.set_ylabel("Frontal Lobe Size (mm)", fontsize=12)
ax.set_title("FL vs CL: Data, MVN Contour, and Eigenvectors",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**What to notice:**

- The **red arrow** (v₁) points along the long axis of the ellipse — the direction of most variability. This captures "overall body size."
- The **blue arrow** (v₂) points along the short axis — the direction of least variability. This captures residual "shape variation."
- The first eigenvector explains the vast majority of the variance — most of what distinguishes these two measurements is simply how big the crab is.

Now: what happens when we add a third variable?


### TODO 3d

Compute the 3×3 covariance matrix for FL, CL, and RW. Compare the off-diagonal entries — which pair has the weakest covariance?


In [ ]:
# TODO 3d: 3x3 covariance matrix
trio_data = crabs[["FL", "CL", "RW"]]
cov_3d = ...

print("Covariance matrix (FL, CL, RW):")
print(cov_3d.round(4))

print(f"\nOff-diagonal entries:")
print(f"  Cov(FL, CL) = {cov_3d.loc['FL','CL']:.4f}")
print(f"  Cov(FL, RW) = {cov_3d.loc['FL','RW']:.4f}")
print(f"  Cov(CL, RW) = {cov_3d.loc['CL','RW']:.4f}")

### TODO 3e

Compute the eigenvalues and eigenvectors of the 3×3 covariance matrix. Compare the variance proportions to the 2D case.

In 2D, the first eigenvector captured almost everything. Does that change when we add RW?


In [ ]:
# TODO 3e: Eigendecomposition in 3D
Sigma_3d = ...
eigenvalues_3d, eigenvectors_3d = ...

# Reverse to descending order
eigenvalues_3d = eigenvalues_3d[::-1]
eigenvectors_3d = eigenvectors_3d[:, ::-1]

prop_first_3d = ...

print("3D Eigendecomposition (FL, CL, RW):")
print("-" * 50)
trio_vars = ["FL", "CL", "RW"]
for i in range(3):
    prop = eigenvalues_3d[i] / eigenvalues_3d.sum()
    vec_str = ", ".join(
        f"{trio_vars[j]}: {eigenvectors_3d[j,i]:+.4f}"
        for j in range(3))
    print(f"  λ_{i+1} = {eigenvalues_3d[i]:8.4f}  "
          f"({prop:5.1%})  v_{i+1} = [{vec_str}]")

print(f"\n--- Comparison ---")
print(f"  2D (FL, CL):     1st eigenvector explains "
      f"{prop_first_2d:.1%}")
print(f"  3D (FL, CL, RW): 1st eigenvector explains "
      f"{prop_first_3d:.1%}")

### Visualize: Eigenvectors in 3D

In 2D, we drew eigenvectors as arrows on a scatter plot with an MVN contour ellipse. We can do the same in 3D — the ellipse becomes an **ellipsoid**, and we get three eigenvector arrows instead of two.

Each arrow is scaled by the standard deviation along that direction $\sqrt{\lambda_i}$ so its length reflects how much variance that eigenvector captures.


In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Data points
ax.scatter(crabs["FL"], crabs["CL"], crabs["RW"],
           c="steelblue", alpha=0.25, s=20,
           edgecolor="white", linewidth=0.2)

# Center of the distribution
mu_3d = trio_data.mean().values  # [FL, CL, RW]
ax.scatter(*mu_3d, color="black", s=120, marker="x",
           linewidths=3, zorder=10)

# Eigenvector arrows scaled by sqrt(eigenvalue)
colors_ev_3d = ["#F44336", "#2196F3", "#FFC107"]
ev_labels = []

for i in range(3):
    vec = eigenvectors_3d[:, i]
    vec_scaled = vec * np.sqrt(eigenvalues_3d[i]) * 2.5
    prop = eigenvalues_3d[i] / eigenvalues_3d.sum()

    for sign in [1, -1]:
        ax.quiver(mu_3d[0], mu_3d[1], mu_3d[2],
                  sign * vec_scaled[0],
                  sign * vec_scaled[1],
                  sign * vec_scaled[2],
                  color=colors_ev_3d[i],
                  arrow_length_ratio=0.15,
                  linewidth=2.5)

    ev_labels.append(f"v{i+1}: {prop:.1%} of variance")

ax.set_xlabel("FL (mm)", fontsize=11, labelpad=8)
ax.set_ylabel("CL (mm)", fontsize=11, labelpad=8)
ax.set_zlabel("RW (mm)", fontsize=11, labelpad=2)
ax.set_title("3D Eigenstructure: Data + Eigenvectors",
             fontsize=13, fontweight="bold", pad=15)

for i in range(3):
    ax.plot([], [], color=colors_ev_3d[i], linewidth=3,
            label=ev_labels[i])
ax.legend(fontsize=10, loc="upper left")

ax.view_init(elev=10, azim=115)

plt.tight_layout()
plt.show()

### TODO 3f

Look at the 3D plot above (try changing `view_init` angles if you want to rotate it). Answer:

- The **red arrow** (v₁) points in what direction relative to all three axes? What does this mean about the crabs?
- The **blue arrow** (v₂) is shorter and points in a different direction. What contrast does it capture?
- The **yellow arrow** (v₃) is the shortest. What remaining variation does it represent?
- Compare the arrow lengths. How does this connect to the eigenvalue proportions you computed in TODO 3e?


*Your answer here.*


### From 3D to Projections

The 3D plot shows the full picture, but it's hard to read precisely — depth perception is limited on a flat screen. In practice, we often look at **2D projections**: views of the data from different angles, like viewing a sculpture from the front, side, and top.

Each projection collapses one axis, so the third variable is encoded as **color** instead of position.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

projections = [
    ("CL", "FL", "RW", "Front View: CL vs FL\n(colored by RW)"),
    ("CL", "RW", "FL", "Side View: CL vs RW\n(colored by FL)"),
    ("RW", "FL", "CL", "Top View: RW vs FL\n(colored by CL)"),
]

for ax, (xvar, yvar, cvar, title) in zip(axes, projections):
    sc = ax.scatter(crabs[xvar], crabs[yvar],
                    c=crabs[cvar], cmap="viridis",
                    alpha=0.6, s=35, edgecolor="white",
                    linewidth=0.3)
    ax.set_xlabel(f"{xvar} (mm)", fontsize=12)
    ax.set_ylabel(f"{yvar} (mm)", fontsize=12)
    ax.set_title(title, fontsize=12, fontweight="bold")
    plt.colorbar(sc, ax=ax, label=f"{cvar} (mm)",
                 shrink=0.85)
    ax.grid(alpha=0.3)

plt.suptitle("Three Views of (FL, CL, RW)",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

**What to notice:**

- The **front view** (CL vs FL) is the familiar plot from Module 2. The color gradient shows that RW tracks body size but with visible scatter — crabs of similar length can have different widths.
- The **side view** (CL vs RW) reveals more spread. The color gradient is smooth but the scatter is wider, consistent with RW's lower correlation.
- In all three views, you can see hints of **two elongated clusters**. There's structure beyond just "big vs. small." We'll come back to this in Part 9.


## Part 4: Conditioning — Slicing the Ellipse (and the Ellipsoid)

In lecture 3.4, you learned the block conditioning formula. If $$\begin{pmatrix} X_1 \\ X_2 \end{pmatrix}$$ is jointly normal, then:

$$\mu_{2|1} = \mu_2 + \Sigma_{21}\,\Sigma_{11}^{-1}\,(x_1 - \mu_1)$$

$$\Sigma_{2|1} = \Sigma_{22} - \Sigma_{21}\,\Sigma_{11}^{-1}\,\Sigma_{12}$$

We'll apply this first in 2D — predicting FL from CL — then in 3D, predicting FL from both CL and RW. Same formula, bigger matrices, better predictions.


### TODO 4a: Build Your Own Normal PDF

Every density curve in this lab uses the univariate normal PDF. Instead of importing it from a library, write it yourself from the formula you know from lecture:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

Complete the function below. It should accept arrays for $x$ so you can evaluate it over a grid of values at once. NumPy handles this automatically — `np.exp()` and `**` work element-wise on arrays.

In [ ]:
# TODO 4a: Complete this function
def normal_pdf(x, mu, sigma):
    """Evaluate the univariate normal density."""
    return ...

# Quick check: standard normal at x = 0 should be ≈ 0.3989
print(f"normal_pdf(0, 0, 1) = {normal_pdf(0, 0, 1):.4f}")

### TODO 4a: Conditioning in 2D

Use the block formula to predict FL for a crab whose CL = 40 mm. Let $X_1 = \text{CL}$ (observed) and $X_2 = \text{FL}$ (to predict).

**Identify the blocks** from the 2×2 covariance matrix:
- $\Sigma_{11}$ = Var(CL)
- $\Sigma_{22}$ = Var(FL)
- $\Sigma_{21}$ = Cov(FL, CL)


In [ ]:
# TODO 4a: Block conditioning in 2D
mu_FL = pair_data["FL"].mean()
mu_CL = pair_data["CL"].mean()

Sigma_22_2d = cov_2d.loc["FL", "FL"]  # Var(FL)
Sigma_11_2d = cov_2d.loc["CL", "CL"]  # Var(CL)
Sigma_21_2d = cov_2d.loc["FL", "CL"]  # Cov(FL, CL)

# Observe CL = 40
cl_observed = 40

# Conditional mean: μ₂ + Σ₂₁/Σ₁₁ * (x₁ - μ₁)
mu_FL_given_CL = ...

# Conditional variance: Σ₂₂ - Σ₂₁²/Σ₁₁
var_FL_given_CL = ...

rho_FL_CL = corr_matrix.loc["FL", "CL"]

print("2D Conditioning: FL | CL = 40")
print("=" * 45)
print(f"  Marginal:    E[FL] = {mu_FL:.2f}, "
      f"Var(FL) = {Sigma_22_2d:.2f}")
print(f"  Conditional: E[FL | CL=40] = "
      f"{mu_FL_given_CL:.2f}")
print(f"               Var(FL | CL=40) = "
      f"{var_FL_given_CL:.2f}")
print(f"               Std(FL | CL=40) = "
      f"{np.sqrt(var_FL_given_CL):.2f} mm")
print(f"\n  Slope: Σ₂₁/Σ₁₁ = "
      f"{Sigma_21_2d/Sigma_11_2d:.4f} mm FL per mm CL")
print(f"  ρ = {rho_FL_CL:.4f},  "
      f"ρ² = {rho_FL_CL**2:.4f} — "
      f"{rho_FL_CL**2*100:.1f}% of variance removed")

### Visualize: Slicing the Ellipse in 2D


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: the slice on the scatter plot
ax = axes[0]
ax.scatter(crabs["CL"], crabs["FL"], color="steelblue",
           alpha=0.3, s=25, edgecolor="white", linewidth=0.3)

# Regression line
cl_range = np.linspace(crabs["CL"].min(),
                       crabs["CL"].max(), 100)
fl_pred = mu_FL + (Sigma_21_2d / Sigma_11_2d) * (cl_range
                                                   - mu_CL)
ax.plot(cl_range, fl_pred, color="red", linewidth=2,
        label="Conditional mean E[FL | CL]")

# The slice
ax.axvline(cl_observed, color="gray", linestyle="--",
           linewidth=1.5, alpha=0.7,
           label=f"Condition: CL = {cl_observed}")
ax.plot(cl_observed, mu_FL_given_CL, "ro", markersize=10,
        zorder=5)

# Confidence band
ax.fill_betweenx(
    [mu_FL_given_CL - 2*np.sqrt(var_FL_given_CL),
     mu_FL_given_CL + 2*np.sqrt(var_FL_given_CL)],
    cl_observed - 1.5, cl_observed + 1.5,
    color="red", alpha=0.15)

ax.set_xlabel("Carapace Length (mm)", fontsize=12)
ax.set_ylabel("Frontal Lobe Size (mm)", fontsize=12)
ax.set_title("Conditioning = Slicing the Ellipse",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)

# Right: marginal vs conditional density of FL
ax = axes[1]
fl_grid = np.linspace(crabs["FL"].min() - 5,
                      crabs["FL"].max() + 5, 300)

marginal_pdf = normal_pdf(fl_grid, mu_FL, np.sqrt(Sigma_22_2d))
ax.plot(fl_grid, marginal_pdf, color="steelblue",
        linewidth=2.5, linestyle="--",
        label=f"Marginal: N({mu_FL:.1f}, {Sigma_22_2d:.1f})")
ax.fill_between(fl_grid, marginal_pdf, alpha=0.1,
                color="steelblue")

cond_pdf = normal_pdf(fl_grid, mu_FL_given_CL,
                    np.sqrt(var_FL_given_CL))
ax.plot(fl_grid, cond_pdf, color="red", linewidth=2.5,
        label=f"Conditional: N({mu_FL_given_CL:.1f}, "
              f"{var_FL_given_CL:.1f})")
ax.fill_between(fl_grid, cond_pdf, alpha=0.15, color="red")

ax.set_xlabel("Frontal Lobe Size (mm)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Knowing CL Narrows Your Belief About FL",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

**What to notice:**

- The conditional density (red) is **narrower** than the marginal (blue dashed). Knowing CL reduced your uncertainty about FL.
- The **conditional variance does not depend on the observed value** of CL. Conditioning on CL = 20 or CL = 50 would give a different center but the same width. This is the "width of the slice" property from lecture 3.4.
- The slope $$\Sigma_{21}/\Sigma_{11}$$ tells you: for each additional mm of CL, FL increases by that many mm on average.

Now: what if we observe a second variable?


### TODO 4c: Conditioning in 3D

Now observe **two** variables: CL = 40 mm and RW = 15 mm. Use the same block formula to predict FL.

**Partition:** Let $X_1 = \begin{pmatrix} \text{CL} \\ \text{RW} \end{pmatrix}$ (observed, 2×1) and $X_2 = \text{FL}$ (to predict, scalar).

Now $\Sigma_{11}$ is 2×2, $\Sigma_{21}$ is 1×2, and the formula requires a matrix inverse instead of scalar division. But the structure is identical.

*Hint:* Keep $\Sigma_{21}$ and $\Sigma_{12}$ as 1D arrays (shape `(2,)`) so the matrix multiplications produce scalars directly.


In [ ]:
# TODO 4c: Block conditioning in 3D
mu_trio = trio_data.mean().values  # [FL, CL, RW]
mu_FL_3 = mu_trio[0]
mu_X1 = mu_trio[1:]  # [CL, RW], shape (2,)

# Build the blocks from the 3x3 covariance matrix
# Order in cov_3d is [FL, CL, RW]
# X2 = FL (index 0), X1 = [CL, RW] (indices 1, 2)
Sigma_22_3d = Sigma_3d[0, 0]           # scalar
Sigma_21_3d = Sigma_3d[0, 1:]          # shape (2,)
Sigma_12_3d = Sigma_3d[1:, 0]          # shape (2,)
Sigma_11_3d = Sigma_3d[1:, 1:]         # shape (2, 2)

# Observe CL = 40, RW = 15
x1_observed_3d = np.array([40.0, 15.0])

# Σ₁₁⁻¹
Sigma_11_inv = ...

# Conditional mean: μ₂ + Σ₂₁ Σ₁₁⁻¹ (x₁ - μ₁)
diff = x1_observed_3d - mu_X1
mu_FL_given_CL_RW = ...

# Conditional variance: Σ₂₂ - Σ₂₁ Σ₁₁⁻¹ Σ₁₂
var_FL_given_CL_RW = ...

# Convert to plain floats
mu_FL_given_CL_RW = float(mu_FL_given_CL_RW)
var_FL_given_CL_RW = float(var_FL_given_CL_RW)

print("3D Conditioning: FL | CL = 40, RW = 15")
print("=" * 50)
print(f"  Conditional: E[FL | CL=40, RW=15] = "
      f"{mu_FL_given_CL_RW:.2f}")
print(f"               Var(FL | CL=40, RW=15) = "
      f"{var_FL_given_CL_RW:.2f}")
print(f"               Std(FL | CL=40, RW=15) = "
      f"{np.sqrt(var_FL_given_CL_RW):.2f} mm")

### TODO 4d: The Payoff — More Observations, Less Uncertainty

Compare the three levels of knowledge about FL:
1. **Nothing observed** — the marginal variance
2. **CL observed** — conditioning on one variable
3. **CL and RW observed** — conditioning on two variables

How much does each additional observation reduce your uncertainty?


In [ ]:
# TODO 4d: Variance reduction comparison

# Hint: some of these values were computed in previous steps
var_marginal = ... # var(FL)
var_after_CL = ... # var(FL | CL)
var_after_CL_RW = ... # var(FL | CL, RW)

reduction_CL = 1 - var_after_CL / var_marginal
reduction_CL_RW = 1 - var_after_CL_RW / var_marginal
additional_from_RW = 1 - var_after_CL_RW / var_after_CL

print("Progressive Variance Reduction for FL")
print("=" * 55)
print(f"  Var(FL)                   = {var_marginal:8.2f}  "
      f"(know nothing)")
print(f"  Var(FL | CL=40)           = {var_after_CL:8.2f}  "
      f"({reduction_CL:.1%} removed)")
print(f"  Var(FL | CL=40, RW=15)    = {var_after_CL_RW:8.2f}  "
      f"({reduction_CL_RW:.1%} removed)")
print(f"\n  CL alone removed {reduction_CL:.1%} of "
      f"marginal variance.")
print(f"  Adding RW removed an additional "
      f"{additional_from_RW:.1%} of remaining variance.")

In [ ]:
# Visualize variance reduction
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: bar chart of variances
ax = axes[0]
labels = ["Marginal\nVar(FL)",
          "After CL\nVar(FL|CL)",
          "After CL+RW\nVar(FL|CL,RW)"]
values = [var_marginal, var_after_CL, var_after_CL_RW]
colors = ["steelblue", "#F44336", "#FF9800"]

bars = ax.bar(labels, values, color=colors, width=0.6,
              edgecolor="white", linewidth=1.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height(),
            f"{val:.2f}", ha="center", va="bottom",
            fontsize=12, fontweight="bold")

ax.set_ylabel("Variance", fontsize=12)
ax.set_title("Each Observation Reduces Uncertainty",
             fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

# Right: three densities overlaid
ax = axes[1]
fl_grid = np.linspace(crabs["FL"].min() - 5,
                      crabs["FL"].max() + 5, 300)

pdf_m = normal_pdf(fl_grid, mu_FL, np.sqrt(var_marginal))
ax.plot(fl_grid, pdf_m, color="steelblue", linewidth=2.5,
        linestyle="--", label="Marginal")
ax.fill_between(fl_grid, pdf_m, alpha=0.08,
                color="steelblue")

pdf_cl = normal_pdf(fl_grid, mu_FL_given_CL,
                  np.sqrt(var_after_CL))
ax.plot(fl_grid, pdf_cl, color="#F44336", linewidth=2.5,
        label="After observing CL")
ax.fill_between(fl_grid, pdf_cl, alpha=0.12,
                color="#F44336")

pdf_clrw = normal_pdf(fl_grid, mu_FL_given_CL_RW,
                    np.sqrt(var_after_CL_RW))
ax.plot(fl_grid, pdf_clrw, color="#FF9800", linewidth=2.5,
        label="After observing CL + RW")
ax.fill_between(fl_grid, pdf_clrw, alpha=0.15,
                color="#FF9800")

ax.set_xlabel("Frontal Lobe Size (mm)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("More Observations → Narrower Belief",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

**What to notice:**

- Each additional observation **narrows** the conditional distribution. More information → less uncertainty.
- CL removes most of the variance because it's highly correlated with FL. RW provides **additional** reduction because it carries information CL doesn't.
- The conditional variance still **does not depend on the observed values**. If we had conditioned on CL = 30 and RW = 10, the width would be the same — only the center shifts.
- The block formula is identical in 2D and 3D. The only difference is that $\Sigma_{11}$ is a scalar in 2D and a 2×2 matrix in 3D. **Same idea, bigger matrices.** This is how conditioning scales.


## Part 5: Conditioning on Sex

In Module 2, you compared conditional PMFs (bar charts) for males and females. Now let's do the same with continuous distributions.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sns.kdeplot(data=crabs, x="ratio", hue="sex",
            fill=True, alpha=0.3, linewidth=2.5,
            palette={"M": "steelblue", "F": "coral"}, ax=ax)

ax.set_xlabel("FL / CL Ratio", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Conditional Density of Ratio by Sex",
             fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

### TODO 5

Compute the conditional means and standard deviations of the ratio for males and females separately.


In [ ]:
# TODO 5: Your code here
males = ...
females = ...

E_ratio_male = ...
E_ratio_female = ...
std_ratio_male = ...
std_ratio_female = ...

print("Conditional statistics:")
print(f"  E[ratio | Male]:   {E_ratio_male:.4f}"
      f"  (std = {std_ratio_male:.4f})")
print(f"  E[ratio | Female]: {E_ratio_female:.4f}"
      f"  (std = {std_ratio_female:.4f})")
print(f"  Difference in means: "
      f"{E_ratio_male - E_ratio_female:.4f}")

**What to notice:**

These are the **continuous** analogs of the conditional expectations you computed from the PMF in Module 2. The values should be similar but not identical — you no longer lose information to binning, and you now have 200 crabs instead of 100.


## Part 6: Bayesian Estimation — Two Roads to the Same Answer

In lecture 3.5, you saw two ways to estimate a normal unknown from normal data:

1. **John's road:** Apply Bayes' rule. Multiply prior × likelihood. Complete the square. Recognize a normal posterior.
2. **The block form:** Write the unknown and observations as a joint normal vector. Apply the conditioning formula from 3.4.

They must give the same answer — because Bayes' rule on jointly normal variables **is** the block conditioning formula.

Let's verify this on the crab data.


### TODO 6a: Road 1 — The Scalar Formulas

Use the conjugate Normal-Normal formulas to estimate the mean ratio $\mu$.

**Prior:** $\mu_0 = 0.50$, $\sigma_0 = 0.05$

Use the sample standard deviation as a plug-in for $\sigma$.

Recall the posterior formulas:

$$\text{precision posterior} = \frac{1}{\sigma_0^2} + \frac{n}{\sigma^2}$$

$$\mu_n = \sigma_n^2 \left( \frac{\mu_0}{\sigma_0^2} + \frac{n\bar{x}}{\sigma^2} \right)$$


In [ ]:
# TODO 6a: Scalar Bayesian estimation
x_bar = crabs["ratio"].mean()
sigma = crabs["ratio"].std()
n = len(crabs)

# Prior
mu_0 = 0.50
sigma_0 = 0.05

# Posterior via scalar formulas
precision_prior = ...
precision_data = ...
precision_posterior = ...

sigma_n_sq = ...
sigma_n = ...
mu_n = ...

print("Road 1: Scalar formulas (Bayes' rule)")
print("=" * 50)
print(f"  Prior precision:     {precision_prior:.1f}")
print(f"  Data precision:      {precision_data:.1f}")
print(f"  Posterior precision:  {precision_posterior:.1f}")
print(f"  μ_n  = {mu_n:.6f}")
print(f"  σ_n  = {sigma_n:.6f}")

### TODO 6b: Road 2 — The Block Form

Now do the same thing using the block conditioning formula.

The idea: treat $\mu$ and $\bar{X}$ as a jointly normal vector. Under our model, $\bar{X} \mid \mu \sim \mathcal{N}(\mu, \sigma^2/n)$, so the pair $(\mu, \bar{X})$ is jointly normal.

**Step 1:** Compute the joint mean vector and covariance matrix.

$$\text{E}[\mu] = \mu_0, \quad \text{E}[\bar{X}] = \mu_0$$

$$\text{Var}(\mu) = \sigma_0^2, \quad \text{Var}(\bar{X}) = \sigma_0^2 + \sigma^2/n$$

$$\text{Cov}(\mu, \bar{X}) = \sigma_0^2$$

**Step 2:** Apply the block formula with $X_1 = \bar{X}$ (observed) and $X_2 = \mu$ (to estimate).


In [ ]:
# TODO 6b: Block form estimation
var_mu = ...
var_xbar = ...
cov_mu_xbar = ...

print("Joint distribution of (μ, X̄):")
print(f"  E[μ]  = {mu_0}")
print(f"  E[X̄]  = {mu_0}")
print(f"  Var(μ)      = {var_mu:.6f}")
print(f"  Var(X̄)      = {var_xbar:.6f}")
print(f"  Cov(μ, X̄)   = {cov_mu_xbar:.6f}")

# Block conditioning: μ | X̄ = x̄
mu_cond_block = ...
var_cond_block = ...

print(f"\nRoad 2: Block conditioning formula")
print(f"  μ_posterior = {mu_cond_block:.6f}")
print(f"  σ_posterior = {np.sqrt(var_cond_block):.6f}")

print(f"\n--- Comparison ---")
print(f"  Road 1 (scalar):  μ_n = {mu_n:.6f}, "
      f"σ_n = {sigma_n:.6f}")
print(f"  Road 2 (block):   μ_n = {mu_cond_block:.6f}, "
      f"σ_n = {np.sqrt(var_cond_block):.6f}")
roads_match = (np.isclose(mu_n, mu_cond_block)
               and np.isclose(sigma_n,
                              np.sqrt(var_cond_block)))
print(f"  Match: {roads_match}")

**What to notice:**

The two roads give **exactly the same answer**. This is not a coincidence — it's a theorem. When the unknown and the observations are jointly normal, Bayes' rule and block conditioning are the same operation.

Road 1 requires algebra: multiply densities, complete the square. Road 2 requires computing the joint covariance matrix, then plugging into a formula. For one unknown and one observation, the effort is comparable. But you just saw in Part 4 that the block formula scales to multiple observations with no new ideas — just bigger matrices. That's the power of Road 2.


### Visualize: Prior, Likelihood, and Posterior


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x_grid = np.linspace(0.40, 0.56, 500)

# Prior
prior_pdf = normal_pdf(x_grid, mu_0, sigma_0)
ax.plot(x_grid, prior_pdf, color="steelblue", linewidth=2.5,
        linestyle="--", label=f"Prior: N({mu_0}, {sigma_0}²)")
ax.fill_between(x_grid, prior_pdf, alpha=0.1,
                color="steelblue")

# Likelihood shape (scaled for visual comparison)
likelihood_std = sigma / np.sqrt(n)
likelihood_pdf = normal_pdf(x_grid, x_bar, likelihood_std)
scale = prior_pdf.max() / likelihood_pdf.max()
likelihood_scaled = likelihood_pdf * scale
ax.plot(x_grid, likelihood_scaled, color="gray", linewidth=2,
        linestyle=":",
        label=f"Likelihood (centered at x̄={x_bar:.3f})")

# Posterior (scaled for visibility)
posterior_pdf = normal_pdf(x_grid, mu_n, sigma_n)
post_scale = prior_pdf.max() / posterior_pdf.max() * 1.3
posterior_scaled = posterior_pdf * post_scale
ax.plot(x_grid, posterior_scaled, color="red", linewidth=3,
        label=f"Posterior: N({mu_n:.4f}, {sigma_n:.4f}²)")
ax.fill_between(x_grid, posterior_scaled, alpha=0.15,
                color="red")

# Mark means
ax.axvline(mu_0, color="steelblue", linestyle="--",
           linewidth=1, alpha=0.4)
ax.axvline(x_bar, color="gray", linestyle=":",
           linewidth=1, alpha=0.4)
ax.axvline(mu_n, color="red", linestyle="-",
           linewidth=1, alpha=0.4)

ax.set_xlabel("μ (mean ratio)", fontsize=13)
ax.set_ylabel("Density (scaled for comparison)", fontsize=12)
ax.set_title("Bayesian Estimation of Mean Crab Ratio",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.set_yticks([])
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## Part 7: MSE and the Width of the Slice

In lecture 3.5, you learned that the **mean squared error** of the Bayesian estimate is just the posterior variance:

$$\text{MSE} = E\bigl[(\mu - \hat{\mu})^2 \mid \text{data}\bigr] = \sigma_n^2$$

And crucially: **this does not depend on the observed data**. It only depends on the variances and the number of observations.

This is the same fact you've already seen twice in this lab:
- Part 4a: Var(FL | CL) doesn't depend on the observed CL
- Part 4c: Var(FL | CL, RW) doesn't depend on the observed CL or RW

Now you'll see it a third time.


Let's compute the MSE for our prescribed prior and verify that it equals the posterior variance.


In [ ]:
print(f"Posterior variance: σ_n² = {sigma_n_sq:.8f}")
print(f"MSE = σ_n²              = {sigma_n_sq:.8f}")
print(f"\nThese are the same number.")
print(f"\nPosterior std (= √MSE): {sigma_n:.6f}")
print(f"\nThis is the remaining uncertainty after seeing "
      f"{n} observations.")

### TODO 7

Does the MSE depend on the *values* of the data, or only on the *amount* of data and the variances?

Test this: compute the posterior mean for three hypothetical datasets with the same $n$ and $\sigma$ but different sample means. Does the MSE change?


In [ ]:
# TODO 7: MSE independence from observed values
hypothetical_xbars = [0.45, 0.486, 0.55]

print(f"Prior: μ₀ = {mu_0}, σ₀ = {sigma_0}")
print(f"Data:  n = {n}, σ = {sigma:.4f}")
print()

for xb in hypothetical_xbars:
    mn = ...  # compute posterior mean for this x̄
    print(f"  x̄ = {xb:.3f}  →  μ_n = {mn:.6f}"
          f"  →  MSE = {sigma_n_sq:.8f}")

print(f"\nThe posterior mean changes with x̄.")
print(f"The MSE does not. It is always {sigma_n_sq:.8f}.")

**Three instances of the same phenomenon:**

| Context | What's fixed | What changes |
|---|---|---|
| Part 4a: Var(FL \| CL=x) | Width of the slice | Center of the slice |
| Part 4c: Var(FL \| CL=x, RW=y) | Width of the slice | Center of the slice |
| Part 7b: MSE of $\hat{\mu}$ | Posterior variance | Posterior mean |

In the multivariate normal, the **width** of the conditional distribution never depends on **where** you condition. The block formula makes this obvious:

$$\Sigma_{2|1} = \Sigma_{22} - \Sigma_{21}\,\Sigma_{11}^{-1}\,\Sigma_{12}$$

contains no observed values — only variances and covariances.


## Part 8: Prior Sensitivity and Scaling with n

Two important questions about Bayesian estimation:
1. How sensitive is the answer to the prior?
2. How does the posterior improve as you collect more data?


### TODO 8a: Prior Sensitivity

Compute the posterior mean for each of these three priors. All use the same data ($n = 200$, $\bar{x}$ and $\sigma$ from the crab ratios).

| Prior | $\mu_0$ | $\sigma_0$ | Interpretation |
|---|---|---|---|
| Confident, correct | 0.486 | 0.01 | Strongly believes right answer |
| Confident, wrong | 0.55 | 0.01 | Strongly believes wrong answer |
| Vague | 0.50 | 0.20 | Barely committed |


In [ ]:
# TODO 8a: Prior sensitivity
priors = [
    ("Confident, correct", 0.486, 0.01),
    ("Confident, wrong",   0.55,  0.01),
    ("Vague",              0.50,  0.20),
]

posterior_means = {}

print(f"Data: x̄ = {x_bar:.4f}, σ = {sigma:.4f}, n = {n}")
print(f"{'':>25s}  {'μ₀':>6s}  {'σ₀':>6s}  "
      f"{'prior prec':>10s}  {'μ_n':>8s}  {'σ_n':>8s}")
print("-" * 75)

for name, m0, s0 in priors:
    prec_prior = ...
    prec_data = ...
    prec_post = ...

    sn = ...
    mn = ...

    posterior_means[name] = mn

    print(f"{name:>25s}  {m0:>6.3f}  {s0:>6.2f}  "
          f"{prec_prior:>10.0f}  {mn:>8.4f}  {sn:>8.4f}")

In [ ]:
# Visualize all three posteriors
fig, ax = plt.subplots(figsize=(10, 6))

x_grid = np.linspace(0.44, 0.56, 500)
colors = ["#4CAF50", "#F44336", "#FF9800"]

for (name, m0, s0), color in zip(priors, colors):
    prec_prior = 1 / s0**2
    prec_data = n / sigma**2
    prec_post = prec_prior + prec_data

    sn = np.sqrt(1 / prec_post)
    mn = (1 / prec_post) * (m0 / s0**2
                            + n * x_bar / sigma**2)

    pdf = normal_pdf(x_grid, mn, sn)
    ax.plot(x_grid, pdf, color=color, linewidth=2.5,
            label=f"{name}: μ_n={mn:.4f}")
    ax.fill_between(x_grid, pdf, alpha=0.1, color=color)

ax.axvline(x_bar, color="gray", linestyle=":", linewidth=2,
           alpha=0.5, label=f"x̄ = {x_bar:.4f}")
ax.set_xlabel("μ (mean ratio)", fontsize=13)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Prior Sensitivity: Three Priors, Same Data",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAll three posteriors are within "
      f"{max(posterior_means.values()) - min(posterior_means.values()):.4f}"
      f" of each other.")

**What to notice:**

Even the "confident and wrong" prior — which strongly believed the mean ratio was 0.55 — gets pulled close to the data. With 200 observations, the data precision overwhelms even a confident prior.

This is the **self-correcting property** of Bayesian estimation: with enough data, reasonable people with different priors converge to the same answer.


### Scaling with n

Now let's fix the prior at $\mu_0 = 0.50$, $\sigma_0 = 0.05$ and see how the posterior changes as you observe more crabs. Subsample the data at $n = 5, 20, 50, 200$.


In [ ]:
np.random.seed(42)

fig, axes = plt.subplots(1, 4, figsize=(18, 4),
                         sharex=True, sharey=True)

mu_0_fixed = 0.50
sigma_0_fixed = 0.05

sample_sizes = [5, 20, 50, 200]
x_grid = np.linspace(0.40, 0.56, 500)

for ax, ns in zip(axes, sample_sizes):
    # Subsample
    if ns < n:
        subsample = crabs["ratio"].sample(ns,
                                          random_state=42)
    else:
        subsample = crabs["ratio"]

    xb = subsample.mean()

    # Posterior
    prec_pr = 1 / sigma_0_fixed**2
    prec_da = ns / sigma**2
    prec_po = prec_pr + prec_da

    sn_sub = np.sqrt(1 / prec_po)
    mn_sub = (1 / prec_po) * (mu_0_fixed / sigma_0_fixed**2
                              + ns * xb / sigma**2)

    # Prior
    prior_pdf = normal_pdf(x_grid, mu_0_fixed, sigma_0_fixed)
    ax.plot(x_grid, prior_pdf, color="steelblue",
            linewidth=1.5, linestyle="--", alpha=0.5)
    ax.fill_between(x_grid, prior_pdf, alpha=0.05,
                    color="steelblue")

    # Posterior
    post_pdf = normal_pdf(x_grid, mn_sub, sn_sub)
    ax.plot(x_grid, post_pdf, color="red", linewidth=2.5)
    ax.fill_between(x_grid, post_pdf, alpha=0.15,
                    color="red")

    ax.axvline(xb, color="gray", linestyle=":",
               linewidth=1, alpha=0.5)
    ax.set_title(f"n = {ns}\nμ_n = {mn_sub:.4f}\n"
                 f"σ_n = {sn_sub:.4f}", fontsize=10)
    ax.set_xlabel("μ", fontsize=10)

axes[0].set_ylabel("Density", fontsize=11)

plt.suptitle("Posterior Narrows as n Grows "
             "(blue dashed = prior, red = posterior)",
             fontsize=13, fontweight="bold", y=1.08)
plt.tight_layout()
plt.show()

**What to notice:**

- At $n = 5$, the posterior is wide and pulled toward the prior. Data is scarce, so the prior has real influence.
- At $n = 200$, the posterior is extremely narrow and centered on $\bar{x}$. The prior is irrelevant.
- The MSE (= posterior variance) shrinks as $n$ grows. This is the $\sigma^2/n$ scaling from lecture.


## Part 9: What's Still Hidden?

You've now analyzed 200 crabs with three focused measurements. You know the covariance structure in 2D and 3D. You've seen that RW adds information the length measurements miss. Your Bayesian estimate is precise. The block formula and Bayes' rule agree.

But look at this:


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: colored by sex
sns.scatterplot(data=crabs, x="CL", y="FL", hue="sex",
                palette={"M": "steelblue", "F": "coral"},
                alpha=0.6, s=40, ax=axes[0])
axes[0].set_xlabel("Carapace Length (mm)", fontsize=12)
axes[0].set_ylabel("Frontal Lobe Size (mm)", fontsize=12)
axes[0].set_title("What We Know: Sex", fontsize=13,
                   fontweight="bold")

# Right: colored by ratio
scatter = axes[1].scatter(crabs["CL"], crabs["FL"],
                          c=crabs["ratio"], cmap="RdYlGn",
                          alpha=0.6, s=40, edgecolor="white",
                          linewidth=0.3)
axes[1].set_xlabel("Carapace Length (mm)", fontsize=12)
axes[1].set_ylabel("Frontal Lobe Size (mm)", fontsize=12)
axes[1].set_title("What We Don't Know Yet: ???",
                   fontsize=13, fontweight="bold")
plt.colorbar(scatter, ax=axes[1], label="FL/CL Ratio")

plt.tight_layout()
plt.show()

In [ ]:
# Three projections colored by ratio
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

projections = [
    ("CL", "FL", "Front View: CL vs FL"),
    ("CL", "RW", "Side View: CL vs RW"),
    ("RW", "FL", "Top View: RW vs FL"),
]

for ax, (xvar, yvar, title) in zip(axes, projections):
    sc = ax.scatter(crabs[xvar], crabs[yvar],
                    c=crabs["ratio"], cmap="RdYlGn",
                    alpha=0.6, s=35, edgecolor="white",
                    linewidth=0.3)
    ax.set_xlabel(f"{xvar} (mm)", fontsize=12)
    ax.set_ylabel(f"{yvar} (mm)", fontsize=12)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.grid(alpha=0.3)

plt.colorbar(sc, ax=axes[-1], label="FL/CL Ratio",
             shrink=0.85)

plt.suptitle("Three Views Colored by Ratio — "
             "Notice the Bands",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

**Look at the ratio-colored plots.**

The colors form **bands** — crabs with similar ratios cluster together, and these clusters don't perfectly align with sex. The banding is visible in all three projections.

There's another variable about these crabs that we haven't revealed yet. It explains the banding. It explains why the ratio distribution looked possibly bimodal back in Module 1. And it explains structure that sex alone can't account for.

**In Module 4, you'll discover what it is.** You'll apply PCA — the eigendecomposition you just did on 2 and 3 variables — to all five measurements at once. The natural axes in 5D will project the data down to 2D, and the hidden variable will reveal itself.


## Part 10: Choose Your Own Prior (Enrichment)

This part is **not graded** — it's an opportunity to think more deeply about the Bayesian approach.

### Your Task

Choose your own values of $\mu_0$ and $\sigma_0$. Write 2–3 sentences justifying your choice. Then compute the posterior and compare to the prescribed prior from Part 6.


In [ ]:
# Your enrichment work here
# 1. Choose mu_0_own and sigma_0_own
# 2. Write your justification as a comment
# 3. Compute the posterior

mu_0_own = ...
sigma_0_own = ...

# Justification:
# ...

# Compute posterior
# prec_prior_own = ...
# prec_data_own = ...
# prec_post_own = ...
# sigma_n_own = ...
# mu_n_own = ...

# print(f"My prior: N({mu_0_own}, {sigma_0_own}²)")
# print(f"Posterior: N({mu_n_own:.4f}, {sigma_n_own:.4f}²)")
# print(f"Prescribed posterior: N({mu_n:.4f}, {sigma_n:.4f}²)")

If your posterior is very close to the prescribed one, that's expected — with 200 data points, the data dominates almost any reasonable prior. The prior matters most when data is scarce.


## Report Your Results

Run the cell below and enter the values on the course platform. Round as indicated.


In [ ]:
print("=" * 60)
print("MODULE 3 REPORT VALUES")
print("=" * 60)
print(f"R1.  Number of crabs:                     "
      f"{n_crabs}")
print(f"R2.  Most independent variable:            "
      f"{most_independent}")
print(f"R3.  Prop variance, 1st eigvec (2D):       "
      f"{prop_first_2d:.4f}")
print(f"R4.  Prop variance, 1st eigvec (3D):       "
      f"{prop_first_3d:.4f}")
print(f"R5.  Var(FL | CL=40) (2 dec):              "
      f"{var_FL_given_CL:.2f}")
print(f"R6.  Var(FL | CL=40, RW=15) (2 dec):       "
      f"{var_FL_given_CL_RW:.2f}")
print(f"R7.  E[ratio | Male] (4 dec):              "
      f"{E_ratio_male:.4f}")
print(f"R8.  E[ratio | Female] (4 dec):            "
      f"{E_ratio_female:.4f}")
print(f"R9.  Posterior mean, prescribed prior:      "
      f"{mu_n:.4f}")
print(f"R10. Posterior std, prescribed prior:       "
      f"{sigma_n:.4f}")

## Reflection Questions

Discuss these with the course chatbot or on the discussion forum.


**Q1.** The correlation matrix shows all values above 0.88. Does this mean the five variables are "the same"? What does high correlation tell you, and what doesn't it tell you?


**Q2.** Compare the eigendecomposition in 2D (FL, CL) and 3D (FL, CL, RW). What does the second eigenvector capture in each case? Why does adding RW change the story?


**Q3.** In this lab, you saw the "width of the slice" property three times:

- Part 4a: Var(FL | CL=x) doesn't depend on x
- Part 4c: Var(FL | CL=x, RW=y) doesn't depend on x or y
- Part 7b: MSE of the Bayesian estimate doesn't depend on $\bar{x}$

Are these the same phenomenon? Explain using the block formula.


**Q4.** In Part 6, you verified that Bayes' rule (Road 1) and the block conditioning formula (Road 2) give the same posterior. What are the advantages and disadvantages of each approach? When would you prefer one over the other?


**Q5.** In Part 4d, observing RW in addition to CL further reduced Var(FL). Will the reduction always be this large? What property of RW relative to CL determines how much additional information it provides?


**Q6.** Using the MSE formula from Part 7, approximately how many crabs would you need so that your posterior standard deviation is less than 0.001?


**Q7.** We told you there's a hidden variable. Based on everything you've seen — the ratio distribution, the sex differences, the scatter plots, the colored ratio bands in all three projections — what do you think it might be?
